# BirdCLEF+ 2026 — Baseline Submission

Loads a debug-trained CNN (10 species) from an attached dataset, runs inference on the hidden test soundscapes,
writes `submission.csv` to `/kaggle/working/`. Species not covered by the model are filled with a uniform prior.

This is a smoke-test baseline — expected leaderboard ≈ 0.5. Purpose: validate the submission pipeline.


In [ ]:
import os, time, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
import timm
from scipy.ndimage import convolve1d

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Torch :", torch.__version__)


In [ ]:
# ---- Paths on Kaggle ---------------------------------------------------
COMP_DIR    = Path("/kaggle/input/competitions/birdclef-2026")
CKPT_PATH   = Path("/kaggle/input/datasets/harishteens/birdclef-2026-baseline-ckpt/model_debug.pt")
TEST_DIR    = COMP_DIR / "test_soundscapes"
TRAIN_DIR   = COMP_DIR / "train_soundscapes"
OUT_PATH    = Path("/kaggle/working/submission.csv")

print("Competition data exists:", COMP_DIR.exists())
print("Checkpoint exists:       ", CKPT_PATH.exists())
print("Test soundscapes exists: ", TEST_DIR.exists())


In [ ]:
# ---- Load checkpoint ---------------------------------------------------
ckpt          = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
species       = ckpt["species"]
label_to_idx  = ckpt["label_to_idx"]
NUM_CLASSES   = ckpt["num_classes"]
BACKBONE      = ckpt["backbone"]
SR            = ckpt["sr"]
N_MELS        = ckpt["n_mels"]
N_FFT         = ckpt["n_fft"]
HOP_LENGTH    = ckpt["hop_length"]
FMIN, FMAX    = ckpt["fmin"], ckpt["fmax"]
CLIP_SEC      = ckpt["clip_sec"]
N_SAMPLES     = SR * CLIP_SEC

# pretrained=False so it doesn't try to download ImageNet weights (no internet during scoring).
model = timm.create_model(BACKBONE, pretrained=False, in_chans=1, num_classes=NUM_CLASSES)
model.load_state_dict(ckpt["state_dict"])
model = model.to(DEVICE).eval()
print(f"Model loaded: {BACKBONE}  classes={NUM_CLASSES}")


In [ ]:
# ---- Mel transform (must match training) ------------------------------
def wav_to_mel(wav):
    mel    = librosa.feature.melspectrogram(
        y=wav, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0,
    )
    mel_db = librosa.power_to_db(mel, top_db=80)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_db.astype(np.float32)


WINDOW_SEC = CLIP_SEC          # 5
N_WINDOWS  = 60 // WINDOW_SEC  # 12

def file_to_chunks(path):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    target = N_WINDOWS * N_SAMPLES
    if len(wav) < target:
        wav = np.pad(wav, (0, target - len(wav)))
    else:
        wav = wav[:target]
    return wav.reshape(N_WINDOWS, N_SAMPLES).astype(np.float32)


GAUSSIAN_KERNEL = np.array([0.1, 0.2, 0.4, 0.2, 0.1])

def smooth_windows(logits):
    return convolve1d(logits, GAUSSIAN_KERNEL, axis=0, mode="nearest")


@torch.no_grad()
def predict_file(path):
    chunks = file_to_chunks(path)
    mels   = np.stack([wav_to_mel(c) for c in chunks])
    mels   = torch.from_numpy(mels).unsqueeze(1).to(DEVICE)
    logits = model(mels).cpu().numpy()
    logits = smooth_windows(logits)
    return 1.0 / (1.0 + np.exp(-logits))


In [ ]:
# ---- Find test files (fallback to train_soundscapes if running locally) ----
test_files = sorted(TEST_DIR.glob("*.ogg")) if TEST_DIR.is_dir() else []
if not test_files:
    test_files = sorted(TRAIN_DIR.glob("*.ogg"))[:5]
    print(f"[fallback] using {len(test_files)} train soundscapes")
else:
    print(f"Found {len(test_files)} test files")


In [ ]:
# ---- Inference loop ----------------------------------------------------
all_rows, all_probs = [], []
t0 = time.time()
for i, f in enumerate(test_files):
    basename = f.stem
    probs    = predict_file(f)
    end_secs = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC
    for k in range(N_WINDOWS):
        all_rows.append(f"{basename}_{end_secs[k]}")
        all_probs.append(probs[k])
    if (i + 1) % 25 == 0 or i == 0 or i == len(test_files) - 1:
        dt = time.time() - t0
        rate = (i + 1) / max(dt, 1e-9)
        print(f"  [{i+1:4d}/{len(test_files)}] {dt:.1f}s  {rate:.2f} files/s")

all_probs = np.stack(all_probs)
print(f"Inference done: {len(all_rows)} rows in {time.time()-t0:.1f}s")


In [ ]:
# ---- Build submission in the exact column order Kaggle expects --------
sample_sub = pd.read_csv(COMP_DIR / "sample_submission.csv")
all_species_in_order = [c for c in sample_sub.columns if c != "row_id"]

pred_df = pd.DataFrame(all_probs, columns=species)
pred_df.insert(0, "row_id", all_rows)

# Reindex to the full taxonomy; species we didn't train on become NaN -> fill with uniform prior.
sub = pred_df.set_index("row_id").reindex(columns=all_species_in_order)
sub = sub.fillna(1.0 / len(all_species_in_order))
sub = sub.clip(0.0, 1.0).reset_index()

assert list(sub.columns) == list(sample_sub.columns), "Column order mismatch!"
assert sub["row_id"].is_unique, "Duplicate row_id!"
assert not sub.isna().any().any(), "NaN in submission!"

sub.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}  shape={sub.shape}  size={OUT_PATH.stat().st_size/1024:.1f}KB")
sub.head(3)
